# pipeline-cpp-grading-v5

## Dependencies

In [1]:
!pip install "huggingface_hub<1.0.0" "transformers>=4.45.0,<5.0.0" "unsloth" "trl" "pydantic" "datasets" --upgrade -q
!pip install wandb -q
import os
os.environ["WANDB_DISABLED"] = "true"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.3/81.3 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 86.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.1/26.1 MB 69.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 24.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 48.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.3/199.3 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.7/146.7 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

## Import

In [2]:
import torch
import json
import os
import glob
import pathlib
import subprocess
import tempfile
import numpy as np
from collections import Counter
from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments
from datasets import Dataset

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPUs: {torch.cuda.device_count()}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)} ({torch.cuda.get_device_properties(i).total_memory / 1e9:.1f} GB)")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


🦥 Unsloth Zoo will now patch everything to make training faster!


Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Using

PyTorch: 2.11.0+cu128
CUDA available: True
GPUs: 2
  GPU 0: Tesla T4 (15.6 GB)
  GPU 1: Tesla T4 (15.6 GB)


## Dataset

In [3]:
root = pathlib.Path("/kaggle/input/datasets/robviet/cpp-grading-dataset")
if not (root / "task1_grading.json").exists():
    found = glob.glob("/kaggle/input/**/task1_grading.json", recursive=True)
    if found:
        root = pathlib.Path(found[0]).parent
    else:
        raise ValueError("Không tìm thấy dataset!")

print(f"Dataset root: {root}")

with open(root / "task1_grading.json", encoding="utf-8") as f: task1 = json.load(f)
with open(root / "task2_error_taxonomy.json", encoding="utf-8") as f: task2 = json.load(f)
with open(root / "task3_feedback.json", encoding="utf-8") as f: task3 = json.load(f)
with open(root / "exams.json", encoding="utf-8") as f: exams = json.load(f)
with open(root / "label_space.json", encoding="utf-8") as f: label_space = json.load(f)

exam_list = exams.get("exams", exams if isinstance(exams, list) else [])
exam_statements = {e["exam_id"]: e["statement"] for e in exam_list}
task1_map = {s["sample_id"]: s for s in task1.get("samples", [])}
task2_map = {s["sample_id"]: s for s in task2.get("samples", [])}
task3_map = {s["sample_id"]: s for s in task3.get("samples", [])}

print(f"Task1: {len(task1_map)} | Task2: {len(task2_map)} | Task3: {len(task3_map)}")

Dataset root: /kaggle/input/datasets/robviet/cpp-grading-dataset/sample_dataset
Task1: 32 | Task2: 32 | Task3: 32


## EDA 

In [4]:
print("=" * 60)
print("EDA + DATA ISSUES")
print("=" * 60)

# Rubric distribution
rubric_dims = ["compilable", "io_format", "logic", "edge_case", "complexity", "code_quality"]
total_scores = []
for s in task1["samples"]:
    r = s["output"]["rubric"]
    total_scores.append(sum(r.get(d, 0) for d in rubric_dims))

print(f"\nTổng điểm: min={min(total_scores)}, max={max(total_scores)}, mean={np.mean(total_scores):.2f}")
print(f"Phân bố: {dict(Counter(total_scores))}")

# Data issues
issues = []
for sid, s1 in task1_map.items():
    r = s1["output"]["rubric"]
    if r.get("compilable", 0) == 0 and r.get("logic", 0) > 0:
        issues.append(f"[MÂU THUẪN] {sid}: compilable=0 nhưng logic={r['logic']}")
    if r.get("io_format", 0) == 0 and r.get("logic", 0) == 4:
        issues.append(f"[MÂU THUẪN] {sid}: io_format=0 nhưng logic=4")
    c_log = s1["input"].get("compile_log", "")
    if "error" in c_log.lower() and r.get("compilable", 0) == 1:
        issues.append(f"[COMPILER MISMATCH] {sid}: log có 'error' nhưng compilable=1")

print(f"\nTổng issues: {len(issues)}")
for issue in issues:
    print(f" {issue}")

# Task 2 labels check
all_labels = []
for s in task2["samples"]:
    all_labels.extend(s["output"].get("errors", []))
print(f"\nTask2 labels: {len(all_labels)} labels, {len(Counter(all_labels))} loại")
if len(all_labels) == 0:
    print("   LABELS RỖNG -> Cần synthetic labels từ compiler")

EDA + DATA ISSUES

Tổng điểm: min=0, max=10, mean=5.41
Phân bố: {8: 2, 9: 4, 0: 3, 2: 4, 6: 1, 1: 5, 10: 5, 7: 7, 3: 1}

Tổng issues: 2
 [MÂU THUẪN] EX02-S205: io_format=0 nhưng logic=4
 [COMPILER MISMATCH] EX02-S214: log có 'error' nhưng compilable=1

Task2 labels: 0 labels, 0 loại
   LABELS RỖNG -> Cần synthetic labels từ compiler


## TASK 3 -> LEVEL EXTRACTION

In [5]:
def extract_level(sample):
    """Extract level từ nhiều vị trí khác nhau trong JSON."""
    for key in ["level", "feedback_level", "target_level"]:
        for section in ["output", "input", "metadata"]:
            if section in sample and key in sample[section]:
                val = sample[section][key]
                if isinstance(val, int): return val
                if isinstance(val, str) and val.isdigit(): return int(val)
    
    # Fallback: tìm trong text
    output_str = json.dumps(sample.get("output", {}))
    for lvl in [4, 3, 2, 1]:
        if f"level {lvl}" in output_str.lower() or f"level_{lvl}" in output_str.lower():
            return lvl
    
    # Fallback: dựa vào độ dài feedback
    feedback = sample.get("output", {}).get("feedback", "")
    if "```" in feedback or "code" in feedback.lower():
        return 4
    elif len(feedback) > 500:
        return 3
    elif len(feedback) > 200:
        return 2
    else:
        return 1

levels_found = {}
for s in task3["samples"]:
    levels_found[s["sample_id"]] = extract_level(s)

print(f"Phân bố Level: {Counter(levels_found.values())}")
print(f"Tổng samples: {len(levels_found)}")

Phân bố Level: Counter({1: 28, 2: 2, 4: 2})
Tổng samples: 32


## Compiler as Ground Truth

In [6]:
# COMPILER INTEGRATION 
def compile_cpp(code: str, timeout: int = 10) -> dict:
    result = {
        "compilable": False, "compile_log": "", "compile_success": False,
        "warnings": [], "errors": [], "error_types": []
    }
    with tempfile.NamedTemporaryFile(suffix=".cpp", mode="w", delete=False, encoding="utf-8") as f:
        f.write(code)
        cpp_path = f.name
    exe_path = cpp_path.replace(".cpp", ".out")
    try:
        proc = subprocess.run(
            ["g++", "-std=c++17", "-O2", "-Wall", "-o", exe_path, cpp_path],
            capture_output=True, text=True, timeout=timeout
        )
        result["compile_log"] = proc.stderr.strip()
        result["compile_success"] = (proc.returncode == 0)
        result["compilable"] = (proc.returncode == 0)
        for line in proc.stderr.split("\n"):
            ll = line.lower()
            if "error:" in ll:
                result["errors"].append(line.strip())
                if "undeclared" in ll or "not declared" in ll:
                    result["error_types"].append("Lỗi biến/hàm chưa khai báo")
                elif "no match for" in ll or "no matching function" in ll:
                    result["error_types"].append("Lỗi hàm")
                elif "expected" in ll:
                    result["error_types"].append("Lỗi cú pháp")
                elif "cannot convert" in ll or "invalid conversion" in ll:
                    result["error_types"].append("Lỗi kiểu dữ liệu")
                elif "ld returned" in ll or "undefined reference" in ll:
                    result["error_types"].append("Lỗi liên kết (missing main)")
                else:
                    result["error_types"].append("Lỗi biên dịch khác")
            elif "warning:" in ll:
                result["warnings"].append(line.strip())
    except subprocess.TimeoutExpired:
        result["compile_log"] = "COMPILATION TIMEOUT"
        result["errors"].append("Compilation timed out")
    except Exception as e:
        result["compile_log"] = str(e)
    finally:
        for p in (cpp_path, exe_path):
            if os.path.exists(p):
                os.unlink(p)
    return result


def generate_synthetic_labels(compiler_result: dict, rubric: dict, code: str) -> list:
    """Tạo synthetic labels cho Task 2 từ compiler + rubric"""
    labels = []
    if not compiler_result["compile_success"]:
        labels.append("Lỗi biên dịch")
        for et in compiler_result["error_types"]:
            if et not in labels:
                labels.append(et)
    if rubric.get("logic", 4) <= 1:
        labels.append("Lỗi logic")
    if rubric.get("edge_case", 2) == 0:
        labels.append("Lỗi xử lý biên")
    if rubric.get("complexity", 1) == 0:
        labels.append("Lỗi thuật toán")
    if rubric.get("io_format", 1) == 0 and compiler_result["compile_success"]:
        labels.append("Lỗi I/O")
    code_lower = code.lower()
    if compiler_result["compile_success"]:
        if "while(true)" in code_lower or "while(1)" in code_lower:
            labels.append("Lỗi vòng lặp")
        if ("new " in code_lower or "malloc" in code_lower) and "delete" not in code_lower:
            labels.append("Lỗi bộ nhớ")
    if not labels:
        labels.append("Không có lỗi")
    return list(set(labels))


# Chạy compiler cho tất cả samples
print("Chạy compiler cho tất cả samples...")
compiler_results = {}
synthetic_task2_labels = {}
for i, (sid, s1) in enumerate(task1_map.items()):
    code_path = root / s1["input"]["code_file"]
    if not code_path.exists():
        continue
    code = code_path.read_text(encoding="utf-8")
    cr = compile_cpp(code)
    compiler_results[sid] = cr
    synthetic_task2_labels[sid] = generate_synthetic_labels(cr, s1["output"]["rubric"], code)
    status = "PASS" if cr["compile_success"] else "FAIL"
    print(f"  [{i+1:2d}/32] {sid}: {status}")

print(f"\nCompiler: {sum(1 for v in compiler_results.values() if v['compile_success'])} PASS / "
      f"{sum(1 for v in compiler_results.values() if not v['compile_success'])} FAIL")

Chạy compiler cho tất cả samples...
  [ 1/32] EX01-S101: PASS
  [ 2/32] EX01-S102: PASS
  [ 3/32] EX01-S103: FAIL
  [ 4/32] EX01-S104: PASS
  [ 5/32] EX01-S105: FAIL
  [ 6/32] EX01-S106: PASS
  [ 7/32] EX01-S107: PASS
  [ 8/32] EX01-S108: FAIL
  [ 9/32] EX01-S109: FAIL
  [10/32] EX01-S110: PASS
  [11/32] EX01-S111: FAIL
  [12/32] EX01-S112: PASS
  [13/32] EX01-S113: PASS
  [14/32] EX01-S114: FAIL
  [15/32] EX01-S115: PASS
  [16/32] EX02-S201: PASS
  [17/32] EX02-S202: PASS
  [18/32] EX02-S203: FAIL
  [19/32] EX02-S204: PASS
  [20/32] EX02-S205: PASS
  [21/32] EX02-S206: PASS
  [22/32] EX02-S207: PASS
  [23/32] EX02-S208: PASS
  [24/32] EX02-S209: PASS
  [25/32] EX02-S210: PASS
  [26/32] EX02-S211: PASS
  [27/32] EX02-S212: FAIL
  [28/32] EX02-S213: PASS
  [29/32] EX02-S214: FAIL
  [30/32] EX02-S215: PASS
  [31/32] EX02-S216: PASS
  [32/32] EX02-S217: FAIL

Compiler: 22 PASS / 10 FAIL


## Train/Val

In [7]:
# TRAIN/VAL 24/8
import random
random.seed(42)

all_samples = []
for sid, s1 in task1_map.items():
    code_path = root / s1["input"]["code_file"]
    if not code_path.exists():
        continue
    code = code_path.read_text(encoding="utf-8")
    statement = exam_statements.get(s1["input"]["exam_id"], "")
    rubric = s1["output"]["rubric"]
    cr = compiler_results.get(sid, {"compile_success": True, "compile_log": "", "errors": [], "warnings": []})
    errors = synthetic_task2_labels.get(sid, ["Không có lỗi"])
    
    compile_log = cr["compile_log"] if cr["compile_log"] else "Không có lỗi biên dịch"
    compile_status = "COMPILE SUCCESS" if cr["compile_success"] else "COMPILE FAILED"
    
    if cr["compile_success"]:
        rationale = "Code biên dịch thành công với g++ -std=c++17."
        if cr["warnings"]:
            rationale += f" Có {len(cr['warnings'])} warning."
    else:
        main_error = cr["errors"][0][:100] if cr["errors"] else "unknown"
        rationale = f"Code KHÔNG biên dịch được. Lỗi chính: {main_error}. Do đó logic=0, edge_case=0."
    
    corrected_rubric = rubric.copy()
    if not cr["compile_success"]:
        corrected_rubric["compilable"] = 0
        corrected_rubric["logic"] = 0
        corrected_rubric["edge_case"] = 0
    
    target = {"rubric": corrected_rubric, "errors": errors}
    
    all_samples.append({
        "sample_id": sid, "statement": statement, "code": code,
        "compile_log": compile_log, "compile_status": compile_status,
        "test_report": s1["input"].get("test_report", "Không có"),
        "target": json.dumps(target, ensure_ascii=False),
        "rationale": rationale, "rubric": corrected_rubric, "errors": errors,
    })

# Split stratified
compilable_samples = [s for s in all_samples if s["rubric"]["compilable"] == 1]
non_compilable_samples = [s for s in all_samples if s["rubric"]["compilable"] == 0]
random.shuffle(compilable_samples)
random.shuffle(non_compilable_samples)

n_val_comp = min(6, len(compilable_samples) // 4)
n_val_noncomp = min(2, len(non_compilable_samples) // 4)
val_samples = compilable_samples[:n_val_comp] + non_compilable_samples[:n_val_noncomp]
train_samples = compilable_samples[n_val_comp:] + non_compilable_samples[n_val_noncomp:]

if len(train_samples) > 24:
    val_samples += train_samples[24:]
    train_samples = train_samples[:24]

print(f"Train: {len(train_samples)} | Val: {len(val_samples)}")
print(f"Train compilable: {sum(1 for s in train_samples if s['rubric']['compilable']==1)}")

Train: 24 | Val: 8
Train compilable: 17


## Prompts

In [8]:
# PROMPTS -> LLM-as-a-Judge + JEV style
def format_judge_prompt(sample):
    input_text = (
        f"### Đề bài:\n{sample['statement']}\n\n"
        f"### Code sinh viên:\n```cpp\n{sample['code']}\n```\n\n"
        f"### Kết quả biên dịch thực tế (g++ -std=c++17):\n{sample['compile_status']}\n"
        f"Log: {sample['compile_log'][:500]}\n\n"
        f"### Test Report:\n{sample['test_report']}"
    )
    text = (
        f"<|im_start|>system\n"
        f"You are an expert C++ code grader acting as a JUDGE. "
        f"You have ACTUAL COMPILATION RESULTS as ground truth - trust them above all.\n"
        f"Grading rubric:\n"
        f"- compilable (0-1): Does code compile? Use compiler result.\n"
        f"- io_format (0-1): Correct input/output format?\n"
        f"- logic (0-4): Algorithm correctness. IF compilable=0, this MUST be 0.\n"
        f"- edge_case (0-2): Handles edge cases? IF compilable=0, this MUST be 0.\n"
        f"- complexity (0-1): Appropriate time/space complexity?\n"
        f"- code_quality (0-1): Clean, readable code?\n\n"
        f"Error taxonomy (multi-label): Lỗi biên dịch, Lỗi logic, Lỗi vòng lặp, "
        f"Lỗi hàm, Lỗi I/O, Lỗi bộ nhớ, Lỗi kiểu dữ liệu, Lỗi thuật toán, "
        f"Lỗi xử lý biên, Lỗi cú pháp, Không có lỗi\n\n"
        f"Output ONLY valid JSON: {{\"rubric\": {{...}}, \"errors\": [...]}}<|im_end|>\n"
        f"<|im_start|>user\n{input_text}<|im_end|>\n"
        f"<|im_start|>assistant\n"
        f"Rationale: {sample['rationale']}\n\n"
        f"Final JSON:\n{sample['target']}<|im_end|>"
    )
    return text

train_texts = [format_judge_prompt(s) for s in train_samples]
val_texts = [format_judge_prompt(s) for s in val_samples]
train_dataset = Dataset.from_dict({"text": train_texts})
val_dataset = Dataset.from_dict({"text": val_texts})
print(f"Train dataset: {len(train_dataset)} | Val dataset: {len(val_dataset)}")

Train dataset: 24 | Val dataset: 8


## LoRA

In [9]:
# LoRA
MODEL_NAME = "unsloth/Qwen2.5-Coder-7B-Instruct"
MAX_SEQ_LENGTH = 4096

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable_params:,} / {total_params:,} ({100*trainable_params/total_params:.2f}%)")

==((====))==  Unsloth 2026.9.14: Fast Qwen2 patching. Transformers: 4.57.6.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
total weights      : 4.308 GiB
no_split classes   : ['Qwen2DecoderLayer']
output head        : lm_head -> cuda:1
head headroom      : 0.359 GiB
activation reserve : 10.632 GiB requested
  cuda:0  budget  12.95 GiB  weights  2.316 GiB  free 10.636 GiB  reserve 10.632 GiB
  cuda:1  budget  12.98 GiB  weights  1.992 GiB  free 10.987 GiB  reserve 10.466 GiB   <- output head
note: Bnb4BitHfQuantizer.adjust_max_memory lowered the budget on cuda:0 14.391 -> 12.952 GiB, cuda:1 14.421 -> 12.979 GiB (memory the quantiser keeps for its own load-time buf

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.9.14 patched 28 layers with 0 QKV layers, 0 O layers and 0 MLP layers. The fused LoRA kernels were skipped because lora_dropout = 0.05, which is why the counts are zero. Training is unaffected.


Trainable: 40,370,176 / 4,393,342,464 (0.92%)


## Train 

In [10]:
import os
os.environ["WANDB_DISABLED"] = "true"

MAX_STEPS = 90

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        per_device_eval_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        max_steps=MAX_STEPS,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=5,
        eval_strategy="steps",
        eval_steps=15,
        save_strategy="steps",
        save_steps=30,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=42,
        output_dir="qwen25_coder_grader_v3",
        report_to="none",
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
    ),
)
trainer.train()

print(f"\nFinal train loss: {trainer.state.log_history[-1].get('train_loss', 'N/A')}")
eval_losses = [e for e in trainer.state.log_history if "eval_loss" in e]
if eval_losses:
    best = min(eval_losses, key=lambda x: x["eval_loss"])
    print(f"Best eval loss: {best['eval_loss']:.4f} (step {best['step']})")

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/24 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/8 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 2
   \\   /|    Num examples = 24 | Num Epochs = 30 | Total steps = 90
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 40,370,176 of 7,655,986,688 (0.53% trained)


Unsloth: Will smartly offload gradients to save VRAM!


/usr/local/lib/python3.13/dist-packages/unsloth/import_fixes.py:5429: UserWarning: 'has_cuda' is deprecated, please use 'torch.backends.cuda.is_built()'
  return original(name)
/usr/local/lib/python3.13/dist-packages/unsloth/import_fixes.py:5429: UserWarning: 'has_cudnn' is deprecated, please use 'torch.backends.cudnn.is_available()'
  return original(name)
/usr/local/lib/python3.13/dist-packages/unsloth/import_fixes.py:5429: UserWarning: 'has_mps' is deprecated, please use 'torch.backends.mps.is_built()'
  return original(name)
/usr/local/lib/python3.13/dist-packages/unsloth/import_fixes.py:5429: UserWarning: 'has_mkldnn' is deprecated, please use 'torch.backends.mkldnn.is_available()'
  return original(name)


Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss,Validation Loss
15,0.741500,0.468685
30,0.146900,0.186376
45,0.102900,0.171399
60,0.066800,0.177029
75,0.042600,0.190990
90,0.040200,0.194317


Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).



Final train loss: 0.2808121850093206
Best eval loss: 0.1714 (step 45)


## JEV-STYLE INFER

In [11]:
def parse_judge_output(text):
    """Parse JSON từ output model."""
    import re
    start = text.find('{')
    end = text.rfind('}')
    if start != -1 and end != -1 and end > start:
        try:
            return json.loads(text[start:end+1])
        except:
            pass
    match = re.search(r'\{.*\}', text, re.DOTALL)
    if match:
        try:
            return json.loads(match.group())
        except:
            pass
    return None


def jev_style_grade(model, tokenizer, sample, temperature=0.0):
    """
    JEV-style grading: Chấm điểm từng tiêu chí một cách có cấu trúc.
    Khác với generate tự do, JEV buộc model phải "chấm điểm" từng mục.
    """
    prompt = format_judge_prompt(sample)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_SEQ_LENGTH)
    inputs = {k: v.to(model.device if hasattr(model, 'device') else "cuda") for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            temperature=temperature,
            do_sample=False,
            top_p=1.0,
        )
    
    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    parsed = parse_judge_output(response)
    return response, parsed


def ensemble_judges(results_list):
    """Kết hợp kết quả từ nhiều judges."""
    valid = [r for r in results_list if r is not None]
    if not valid:
        return None
    
    rubric_dims = ["compilable", "io_format", "logic", "edge_case", "complexity", "code_quality"]
    ensemble_rubric = {}
    
    for dim in rubric_dims:
        values = [r["rubric"].get(dim, 0) for r in valid if "rubric" in r]
        if not values:
            ensemble_rubric[dim] = 0
            continue
        if dim == "compilable":
            ensemble_rubric[dim] = min(values)  # Conservative
        else:
            ensemble_rubric[dim] = round(np.mean(values))
    
    # Ensemble errors
    error_counts = Counter()
    for r in valid:
        for e in r.get("errors", []):
            error_counts[e] += 1
    threshold = len(valid) / 2
    ensemble_errors = [e for e, c in error_counts.items() if c >= threshold]
    
    return {"rubric": ensemble_rubric, "errors": ensemble_errors}

## Evaluate Vad

In [12]:
print("Chạy LLM-as-a-Judge trên Validation Set...")
FastLanguageModel.for_inference(model)

val_predictions = []
for i, sample in enumerate(val_samples):
    response, parsed = jev_style_grade(model, tokenizer, sample)
    gt_rubric = sample["rubric"]
    gt_errors = sample["errors"]
    
    pred_rubric = parsed["rubric"] if parsed and "rubric" in parsed else {}
    pred_errors = parsed["errors"] if parsed and "errors" in parsed else []
    
    rubric_diff = sum(abs(pred_rubric.get(d, 0) - gt_rubric.get(d, 0)) 
                      for d in ["compilable", "io_format", "logic", "edge_case", "complexity", "code_quality"])
    
    print(f"  [{i+1}/{len(val_samples)}] {sample['sample_id']}: rubric_diff={rubric_diff}, "
          f"pred_errors={len(pred_errors)}, gt_errors={len(gt_errors)}")
    
    val_predictions.append({
        "sample_id": sample["sample_id"],
        "pred_rubric": pred_rubric,
        "gt_rubric": gt_rubric,
        "pred_errors": pred_errors,
        "gt_errors": gt_errors,
        "rubric_diff": rubric_diff,
    })

avg_diff = np.mean([p["rubric_diff"] for p in val_predictions])
exact_match = sum(1 for p in val_predictions if p["rubric_diff"] == 0) / len(val_predictions)
print(f"\nVALIDATION RESULTS:")
print(f"  Average rubric difference: {avg_diff:.2f} (lower is better)")
print(f"  Exact match rate: {exact_match:.1%}")

Chạy LLM-as-a-Judge trên Validation Set...
  [1/8] EX02-S216: rubric_diff=7, pred_errors=0, gt_errors=1
  [2/8] EX01-S110: rubric_diff=8, pred_errors=0, gt_errors=1
  [3/8] EX01-S104: rubric_diff=9, pred_errors=0, gt_errors=1
  [4/8] EX02-S208: rubric_diff=7, pred_errors=0, gt_errors=1
  [5/8] EX02-S201: rubric_diff=10, pred_errors=0, gt_errors=1
  [6/8] EX02-S212: rubric_diff=1, pred_errors=0, gt_errors=5
  [7/8] EX01-S114: rubric_diff=0, pred_errors=0, gt_errors=5
  [8/8] EX02-S214: rubric_diff=3, pred_errors=0, gt_errors=2

VALIDATION RESULTS:
  Average rubric difference: 5.62 (lower is better)
  Exact match rate: 12.5%


In [13]:
def parse_judge_output(text):
    start = text.find('{')
    end = text.rfind('}')
    if start != -1 and end > start:
        try:
            return json.loads(text[start:end+1])
        except:
            pass
    return None

FastLanguageModel.for_inference(model)

val_predictions = []
for i, sample in enumerate(val_samples):
    prompt = format_judge_prompt(sample)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_SEQ_LENGTH)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=512, temperature=0.0, do_sample=False)
    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    parsed = parse_judge_output(response)
    
    gt_rubric = sample["rubric"]
    pred_rubric = parsed.get("rubric", {}) if parsed else {}
    
    rubric_diff = sum(abs(pred_rubric.get(d, 0) - gt_rubric.get(d, 0)) 
                      for d in ["compilable", "io_format", "logic", "edge_case", "complexity", "code_quality"])
    
    val_predictions.append({"sample_id": sample["sample_id"], "rubric_diff": rubric_diff})
    print(f"  [{i+1}/{len(val_samples)}] {sample['sample_id']}: rubric_diff={rubric_diff}")

avg_diff = np.mean([p["rubric_diff"] for p in val_predictions])
exact_match = sum(1 for p in val_predictions if p["rubric_diff"] == 0) / len(val_predictions)
print(f"\nAvg rubric diff: {avg_diff:.2f} (lower is better)")
print(f"Exact match: {exact_match:.1%}")

  [1/8] EX02-S216: rubric_diff=7
  [2/8] EX01-S110: rubric_diff=8
  [3/8] EX01-S104: rubric_diff=9
  [4/8] EX02-S208: rubric_diff=7
  [5/8] EX02-S201: rubric_diff=10
  [6/8] EX02-S212: rubric_diff=1
  [7/8] EX01-S114: rubric_diff=0
  [8/8] EX02-S214: rubric_diff=3

Avg rubric diff: 5.62 (lower is better)
Exact match: 12.5%


## Save 

In [14]:
import shutil

cache_dirs = ["/root/.cache/huggingface/hub", "/kaggle/working/qwen25_coder_grader_v3"]
for d in cache_dirs:
    if os.path.exists(d):
        try: shutil.rmtree(d)
        except: pass

save_path = "lora_adapter_v3"
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

total_size = sum(os.path.getsize(os.path.join(dp, f)) for dp, dn, fn in os.walk(save_path) for f in fn)
print(f"Saved LoRA adapter to '{save_path}' ({total_size / 1e6:.1f} MB)")

!zip -r lora_adapter_v3.zip lora_adapter_v3/
print("Download file 'lora_adapter_v3.zip' từ Output panel")
!df -h /kaggle/working | tail -1

config.json: 0.00B [00:00, ?B/s]

Saved LoRA adapter to 'lora_adapter_v3' (177.4 MB)
  adding: lora_adapter_v3/ (stored 0%)
  adding: lora_adapter_v3/tokenizer.json (deflated 81%)
  adding: lora_adapter_v3/README.md (deflated 65%)
  adding: lora_adapter_v3/vocab.json (deflated 61%)
  adding: lora_adapter_v3/adapter_config.json (deflated 59%)
  adding: lora_adapter_v3/tokenizer_config.json (deflated 89%)
  adding: lora_adapter_v3/special_tokens_map.json (deflated 67%)
  adding: lora_adapter_v3/merges.txt (deflated 57%)
  adding: lora_adapter_v3/chat_template.jinja (deflated 71%)
  adding: lora_adapter_v3/added_tokens.json (deflated 65%)
  adding: lora_adapter_v3/adapter_model.safetensors (deflated 8%)
Download file 'lora_adapter_v3.zip' từ Output panel
/dev/loop2       20G  329M   20G   2% /kaggle/working


## Sumary

In [15]:
print("=" * 60)
print("FINAL SUMMARY")
print("=" * 60)
print(f"Compiler: {sum(1 for v in compiler_results.values() if v['compile_success'])}/32 PASS")
print(f"Val avg rubric diff: {avg_diff:.2f}")
print(f"Val exact match: {exact_match:.1%}")
print(f"Data issues found: {len(issues)}")
for issue in issues[:5]:
    print(f" {issue}")

FINAL SUMMARY
Compiler: 22/32 PASS
Val avg rubric diff: 5.62
Val exact match: 12.5%
Data issues found: 2
 [MÂU THUẪN] EX02-S205: io_format=0 nhưng logic=4
 [COMPILER MISMATCH] EX02-S214: log có 'error' nhưng compilable=1


In [16]:
# Eval + EDA
print("=" * 60)
print("ĐÁNH GIÁ")
print("=" * 60)

# 1. So sánh compiler vs teacher
print("\n1. SO SÁNH COMPILER vs TEACHER:")
mismatches = []
for sid, s1 in task1_map.items():
    teacher_comp = s1["output"]["rubric"].get("compilable", -1)
    compiler_comp = 1 if compiler_results.get(sid, {}).get("compile_success", True) else 0
    if teacher_comp != compiler_comp:
        mismatches.append(f"  {sid}: teacher={teacher_comp}, compiler={compiler_comp}")

print(f"  Mismatches: {len(mismatches)}/32")
for m in mismatches:
    print(m)

# 2. Synthetic Task 2 labels
print(f"\n2. SYNTHETIC TASK 2 LABELS:")
all_syn = [l for v in synthetic_task2_labels.values() for l in v]
print(f"  Tổng labels: {len(all_syn)}")
print(f"  Phân bố: {dict(Counter(all_syn).most_common())}")

# 3. Task 3 levels
print(f"\n3. TASK 3 LEVELS:")
for lvl, cnt in sorted(Counter(levels_found.values()).items()):
    print(f"    Level {lvl}: {cnt} samples")

# 4. Data issues
print(f"\n4. DATA ISSUES:")
issues = []
for sid, s1 in task1_map.items():
    r = s1["output"]["rubric"]
    if r.get("compilable", 0) == 0 and r.get("logic", 0) > 0:
        issues.append(f"  [MÂU THUẪN] {sid}: compilable=0 nhưng logic={r['logic']}")
    if r.get("io_format", 0) == 0 and r.get("logic", 0) == 4:
        issues.append(f"  [MÂU THUẪN] {sid}: io_format=0 nhưng logic=4")
    cr = compiler_results.get(sid, {})
    if not cr.get("compile_success", True) and r.get("compilable", 0) == 1:
        issues.append(f"  [COMPILER MISMATCH] {sid}: compiler FAIL nhưng teacher nói compilable=1")

print(f"  Tổng issues: {len(issues)}")
for issue in issues:
    print(issue)

print(f"\n{'='*60}")
print("DONE!")

ĐÁNH GIÁ

1. SO SÁNH COMPILER vs TEACHER:
  Mismatches: 4/32
  EX01-S105: teacher=1, compiler=0
  EX01-S108: teacher=1, compiler=0
  EX01-S114: teacher=1, compiler=0
  EX02-S214: teacher=1, compiler=0

2. SYNTHETIC TASK 2 LABELS:
  Tổng labels: 86
  Phân bố: {'Không có lỗi': 15, 'Lỗi logic': 14, 'Lỗi xử lý biên': 13, 'Lỗi thuật toán': 12, 'Lỗi biên dịch': 10, 'Lỗi biên dịch khác': 4, 'Lỗi liên kết (missing main)': 4, 'Lỗi biến/hàm chưa khai báo': 4, 'Lỗi I/O': 4, 'Lỗi cú pháp': 3, 'Lỗi bộ nhớ': 2, 'Lỗi hàm': 1}

3. TASK 3 LEVELS:
    Level 1: 28 samples
    Level 2: 2 samples
    Level 4: 2 samples

4. DATA ISSUES:
  Tổng issues: 5
  [COMPILER MISMATCH] EX01-S105: compiler FAIL nhưng teacher nói compilable=1
  [COMPILER MISMATCH] EX01-S108: compiler FAIL nhưng teacher nói compilable=1
  [COMPILER MISMATCH] EX01-S114: compiler FAIL nhưng teacher nói compilable=1
  [MÂU THUẪN] EX02-S205: io_format=0 nhưng logic=4
  [COMPILER MISMATCH] EX02-S214: compiler FAIL nhưng teacher nói compilable